# 03 — Building Characteristics

Aggregates PLUTO building-level statistics per grid cell.

**Data source:** PLUTO CSV (NYC only — `needs_pluto`).

**Output columns:** `cell_id`, `avg_floors`, `avg_yearbuilt`, `building_count`, `total_bldg_area`

**Output file:** `csv/03_building_characteristics.csv`

In [ ]:
# ── Papermill parameters ──────────────────────────────
GRID_CONFIG = "grid.json"

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import math

os.makedirs("csv", exist_ok=True)

with open(GRID_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

if not config["feature_flags"].get("needs_pluto", False):
    print("PLUTO not available — skipping notebook 03.")
    df_grid = pd.read_csv("csv/01_grid_definition.csv", dtype={"cell_id": str})
    df_empty = pd.DataFrame({"cell_id": df_grid["cell_id"]})
    for col in ["avg_floors", "avg_yearbuilt", "building_count", "total_bldg_area"]:
        df_empty[col] = np.nan
    df_empty.to_csv("csv/03_building_characteristics.csv", index=False)
    raise SystemExit("Skipped — needs_pluto=false")

PLUTO_PATH = config["pluto_path"]
BOROUGH_CODES = config["borough_codes"]
BOROUGH_FILTER = config["borough_filter"]
CELL_SIZE_M = config["grid_cell_size_m"]
boro_code_filter = [str(BOROUGH_CODES[b]) for b in BOROUGH_FILTER]
print(f"Loading PLUTO from {PLUTO_PATH}")

In [ ]:
# ── Load PLUTO + grid parameters ──────────────────────
COLS = ["borocode", "numfloors", "yearbuilt", "lotarea", "bldgarea", "numbldgs",
        "latitude", "longitude"]

df_pluto = pd.read_csv(PLUTO_PATH, usecols=COLS)
df_pluto = df_pluto[df_pluto["borocode"].astype(str).isin(boro_code_filter)].copy()

for col in ["numfloors", "yearbuilt", "lotarea", "bldgarea", "numbldgs"]:
    df_pluto[col] = pd.to_numeric(df_pluto[col], errors="coerce")
df_pluto["latitude"] = pd.to_numeric(df_pluto["latitude"], errors="coerce")
df_pluto["longitude"] = pd.to_numeric(df_pluto["longitude"], errors="coerce")
df_pluto = df_pluto.dropna(subset=["latitude", "longitude"]).copy()

df_pluto.loc[df_pluto["yearbuilt"] < 1700, "yearbuilt"] = np.nan
df_pluto.loc[df_pluto["numfloors"] <= 0, "numfloors"] = np.nan

# Grid parameters — must match notebook 01
REF_LAT = df_pluto["latitude"].mean()
LAT_STEP = CELL_SIZE_M / 111_000
LON_STEP = CELL_SIZE_M / (111_000 * math.cos(math.radians(REF_LAT)))
BUFFER = LAT_STEP
LAT_MIN = df_pluto["latitude"].min() - BUFFER
LON_MIN = df_pluto["longitude"].min() - BUFFER

# Assign lots to grid cells
df_pluto["grid_row"] = ((df_pluto["latitude"] - LAT_MIN) / LAT_STEP).astype(int)
df_pluto["grid_col"] = ((df_pluto["longitude"] - LON_MIN) / LON_STEP).astype(int)
df_pluto["cell_id"] = "r" + df_pluto["grid_row"].astype(str).str.zfill(4) + "_c" + df_pluto["grid_col"].astype(str).str.zfill(4)

# Only keep lots in valid grid cells
df_grid = pd.read_csv("csv/01_grid_definition.csv", dtype={"cell_id": str})
valid_cells = set(df_grid["cell_id"])
df_pluto = df_pluto[df_pluto["cell_id"].isin(valid_cells)].copy()
print(f"PLUTO lots in valid grid cells: {len(df_pluto):,}")

In [ ]:
# ── Aggregate per grid cell ────────────────────────────
agg = df_pluto.groupby("cell_id").agg(
    avg_floors=("numfloors", "mean"),
    avg_yearbuilt=("yearbuilt", "mean"),
    total_bldg_area=("bldgarea", "sum"),
    building_count=("numbldgs", "sum"),
).reset_index()

agg["avg_floors"] = agg["avg_floors"].round(1)
agg["avg_yearbuilt"] = agg["avg_yearbuilt"].round(0).astype("Int64")
agg["total_bldg_area"] = agg["total_bldg_area"].round(0)
agg["building_count"] = agg["building_count"].astype(int)

# Ensure all grid cells are present
df_result = df_grid[["cell_id"]].merge(agg, on="cell_id", how="left")
print(f"Aggregated {len(df_result)} cells")
print(df_result.describe().round(1).to_string())

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = "csv/03_building_characteristics.csv"
df_result.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_result)} rows x {df_result.shape[1]} cols)")
df_result.head(10)